# Atığını Tanı — RealWaste Görüntü Sınıflandırma

Bu notebook, proje ZIP'ini yükledikten sonra UCI RealWaste veri setini indirir, modeli eğitir ve değerlendirme görsellerini gösterir. Colab'da **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçmeniz önerilir.

In [ ]:
!pip -q install 'torch>=2.6,<3' 'numpy>=2,<3' 'Pillow>=11,<13' 'matplotlib>=3.9,<4' 'scikit-learn>=1.6,<2' 'transformers>=5,<6'
print('Bağımlılıklar hazır.')

## 1. Proje ZIP'ini yükle
`03` değil, `04-Makine-ve-Derin-Ogrenme-Atigini-Tani.zip` dosyasını seçin.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
archive = Path(next(iter(uploaded)))
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/content/atigini-tani')
project_candidates = list(Path('/content/atigini-tani').rglob('04-atigini-tani-atik-siniflandirma'))
PROJECT = project_candidates[0] if project_candidates else Path('/content/atigini-tani')
%cd {PROJECT}
print('Proje klasörü:', PROJECT)

## 2. Lisanslı veri setini indir
Kaynak: UCI RealWaste, DOI `10.24432/C5SS4G`. UCI kaydı CC BY 4.0, yazar deposu CC BY-NC-SA 4.0 belirtir; proje daha kısıtlayıcı koşulu esas alır.

In [ ]:
from pathlib import Path
import subprocess

data_dir = Path('data/raw/realwaste-main/RealWaste')
if not data_dir.exists():
    subprocess.run(['bash', 'scripts/download_data.sh'], check=True)
print('Veri hazır:', data_dir)

## 3. Modeli eğit ve değerlendir
Aynı seed, split, mimari ve hiperparametrelerle yeniden üretilebilir deney çalışır.

In [ ]:
!python src/train_model.py --data-dir data/raw/realwaste-main/RealWaste --epochs 12 --batch-size 32 --image-size 224 --learning-rate 1e-4 --seed 42

## 4. Sonuç özeti ve kanıt görselleri

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image as DisplayImage

metrics = json.loads(Path('results/metrics.json').read_text())
print(json.dumps(metrics['test'], indent=2, ensure_ascii=False))
for figure in sorted(Path('figures').glob('*.png')):
    print('\n', figure.name)
    display(DisplayImage(filename=str(figure), width=900))